# Classic ML Baselines

This notebook runs the final classic baseline grid across multiple random seeds and creates analysis tables:

- `total_results`: all evaluated combinations across all seeds.
- `per_seed_report_results`: the best validation result per seed and representation family.
- `report_results`: mean/std validation metrics across seeds.

The task predicts sentiment labels `0..4` for mixed English/German product reviews. The validation score is `1 - MAE / 4`.

The final grid contains a majority baseline; sparse BoW/TF-IDF word/character features with Logistic Regression, Linear SVM, Ridge Classifier, and Complement NB; and static embedding features with average, TF-IDF weighted, and mean+max pooling trained with Logistic Regression, Linear SVM, and Ridge Classifier.


In [1]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

import pandas as pd

EXPERIMENT_KIND = "CLASSIC_ML_BASELINES"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = Path("experiments/classic_ml") / f"{RUN_ID}_{EXPERIMENT_KIND}"
EMBEDDING_DIR = Path("experiments/embeddings")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("data/train.csv")
VALIDATION_SIZE = 0.1
SEEDS = [42, 43, 44]
MAX_FEATURES = 250_000
MAX_ITER = 100

embedding_paths = {
    "glove": EMBEDDING_DIR / "glove.6B.300d.txt",
    "fasttext": EMBEDDING_DIR / "fasttext.vec",
}

available_embeddings = {
    name: path for name, path in embedding_paths.items() if path.exists()
}
EXPERIMENT_DIR, SEEDS, available_embeddings


(PosixPath('experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES'),
 [42, 43, 44],
 {'glove': PosixPath('experiments/embeddings/glove.6B.300d.txt'),
  'fasttext': PosixPath('experiments/embeddings/fasttext.vec')})

This notebook is configured for the final classic run: sparse lexical baselines plus dense static-embedding baselines for every seed. Place pretrained static embeddings in `experiments/embeddings/` with the filenames configured above.


In [2]:
if not available_embeddings:
    raise FileNotFoundError("No embedding files found in experiments/embeddings/. Run load_embeddings.ipynb first.")

completed_runs = []
mode = "all"

for seed in SEEDS:
    seed_dir = EXPERIMENT_DIR / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "-m",
        "baselines.classic_ml_baselines",
        "--mode",
        mode,
        "--train-path",
        str(TRAIN_PATH),
        "--output-dir",
        str(seed_dir),
        "--validation-size",
        str(VALIDATION_SIZE),
        "--random-state",
        str(seed),
        "--max-features",
        str(MAX_FEATURES),
        "--max-iter",
        str(MAX_ITER),
    ]
    for name, path in available_embeddings.items():
        cmd.extend([f"--{name}-path", str(path)])

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    completed_runs.append({"seed": seed, "run_dir": seed_dir})

completed_runs


/cluster/courses/cil/envs/envs/text-5060/bin/python -m baselines.classic_ml_baselines --mode all --train-path data/train.csv --output-dir experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/seed_42 --validation-size 0.1 --random-state 42 --max-features 250000 --max-iter 100 --glove-path experiments/embeddings/glove.6B.300d.txt --fasttext-path experiments/embeddings/fasttext.vec


Train examples: 226800
Validation examples: 25200
Class counts: {0: 45360, 1: 45360, 2: 45360, 3: 45360, 4: 45360}
[job] majority__most_frequent | jobs_run: 1/40
[ok] majority__most_frequent: score=0.50000 mae=2.00000 acc=0.20000 macro_f1=0.06667
[job] bow__word_1_3__features | jobs_run: 2/40


[job] bow__word_1_3__logreg | jobs_run: 3/40


/cluster/courses/cil/envs/envs/text-5060/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[ok] bow__word_1_3__logreg: score=0.86758 mae=0.52968 acc=0.56897 macro_f1=0.56825
[job] bow__word_1_3__linear_svm | jobs_run: 4/40


...

/cluster/courses/cil/envs/envs/text-5060/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[ok] fasttext__mean_max__logreg: score=0.80535 mae=0.77861 acc=0.47849 macro_f1=0.47098
[job] fasttext__mean_max__linear_svm | jobs_run: 39/40


[ok] fasttext__mean_max__linear_svm: score=0.81723 mae=0.73107 acc=0.50349 macro_f1=0.48845
[job] fasttext__mean_max__ridge_classifier | jobs_run: 40/40


[ok] fasttext__mean_max__ridge_classifier: score=0.81210 mae=0.75159 acc=0.49651 macro_f1=0.48162
Wrote experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/seed_44/classic_ml_results.csv
Wrote experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/seed_44/classic_ml_best_by_family.csv


[{'seed': 42,
  'run_dir': PosixPath('experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/seed_42')},
 {'seed': 43,
  'run_dir': PosixPath('experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/seed_43')},
 {'seed': 44,
  'run_dir': PosixPath('experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/seed_44')}]

## Total Analysis

In [3]:
result_frames = []
for run in completed_runs:
    frame = pd.read_csv(run["run_dir"] / "classic_ml_results.csv")
    frame.insert(0, "seed", run["seed"])
    result_frames.append(frame)

results = pd.concat(result_frames, ignore_index=True)
results.to_csv(EXPERIMENT_DIR / "classic_ml_results.csv", index=False)

total_results = results.sort_values(
    ["status", "cil_score"], ascending=[False, False]
).reset_index(drop=True)
total_results.to_csv(EXPERIMENT_DIR / "total_analysis.csv", index=False)
total_results


,seed,experiment,family,representation,variant,classifier,status,train_seconds,predict_seconds,accuracy,macro_f1,mae,cil_score,quadratic_weighted_kappa,notes
0,44,tfidf_word__word_1_3__logreg,tfidf_word,"word n-grams=(1, 3)",word_1_3,logreg,ok,50.853062,0.031266,0.598333,0.595693,0.499048,0.875238,0.817048,"feature_seconds=69.57; train_shape=(226800, 25..."
1,42,tfidf_word__word_1_3__logreg,tfidf_word,"word n-grams=(1, 3)",word_1_3,logreg,ok,42.306954,0.023856,0.595595,0.592839,0.501389,0.874653,0.816759,"feature_seconds=60.99; train_shape=(226800, 25..."
2,43,tfidf_word__word_1_3__logreg,tfidf_word,"word n-grams=(1, 3)",word_1_3,logreg,ok,41.535271,0.021415,0.590357,0.585986,0.514643,0.871339,0.810656,"feature_seconds=58.49; train_shape=(226800, 25..."
3,42,tfidf_char__char_wb_3_6__logreg,tfidf_char,"char_wb n-grams=(3, 6)",char_wb_3_6,logreg,ok,159.854112,0.093858,0.583849,0.580399,0.522897,0.869276,0.805958,"feature_seconds=86.26; train_shape=(226800, 25..."
4,44,tfidf_char__char_wb_3_6__logreg,tfidf_char,"char_wb n-grams=(3, 6)",char_wb_3_6,logreg,ok,162.642435,0.079766,0.580476,0.577999,0.527619,0.868095,0.803204,"feature_seconds=95.39; train_shape=(226800, 25..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,43,glove__tfidf_weighted__ridge_classifier,glove,glove tfidf_weighted,tfidf_weighted,ridge_classifier,ok,1.197790,0.015891,0.436984,0.415602,0.905516,0.773621,0.584771,feature_seconds=50.95; vocab_cov=0.3326; token...
89,42,glove__tfidf_weighted__ridge_classifier,glove,glove tfidf_weighted,tfidf_weighted,ridge_classifier,ok,0.894921,0.014813,0.435675,0.414048,0.907183,0.773204,0.586081,feature_seconds=44.59; vocab_cov=0.3329; token...
90,42,majority__most_frequent,majority,constant,most_frequent,dummy_most_frequent,ok,0.009343,0.000206,0.200000,0.066667,2.000000,0.500000,0.000000,Always predicts the most frequent training label.
91,43,majority__most_frequent,majority,constant,most_frequent,dummy_most_frequent,ok,0.008632,0.000298,0.200000,0.066667,2.000000,0.500000,0.000000,Always predicts the most frequent training label.


## Report Analysis

In [4]:
ok_results = results[results["status"] == "ok"].copy()
per_seed_report_results = (
    ok_results.sort_values("cil_score", ascending=False)
    .groupby(["seed", "family"], as_index=False)
    .first()
    .sort_values(["family", "seed"])
    .reset_index(drop=True)
)
per_seed_report_results.to_csv(EXPERIMENT_DIR / "per_seed_report_analysis.csv", index=False)

report_results = (
    per_seed_report_results
    .groupby("family")[["cil_score", "mae", "accuracy", "macro_f1"]]
    .agg(["mean", "std"])
    .reset_index()
)
report_results.columns = [
    column[0] if column[1] == "" else f"{column[0]}_{column[1]}"
    for column in report_results.columns.to_flat_index()
]
report_results = report_results.sort_values("cil_score_mean", ascending=False).reset_index(drop=True)
report_results.to_csv(EXPERIMENT_DIR / "report_analysis.csv", index=False)
report_results


,family,cil_score_mean,cil_score_std,mae_mean,mae_std,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std
0,tfidf_word,0.873743,0.002102,0.505026,0.008410,0.594762,0.004053,0.591506,0.004989
1,tfidf_char,0.868363,0.000813,0.526548,0.003250,0.582090,0.001691,0.578646,0.001536
2,bow,0.866319,0.001166,0.534722,0.004665,0.569167,0.001498,0.566809,0.002357
3,fasttext,0.817563,0.000399,0.729749,0.001596,0.495847,0.007150,0.486907,0.002163
4,glove,0.800017,0.001140,0.799934,0.004561,0.459775,0.003386,0.453601,0.003706
5,majority,0.500000,0.000000,2.000000,0.000000,0.200000,0.000000,0.066667,0.000000


## Analysis Artifacts


In [5]:
analysis_dir = EXPERIMENT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

classic_fastest_table = analysis_dir / "classic_fastest_families.tex"
classic_full_table = analysis_dir / "classic_full_results.tex"
classic_seed_summary_table = analysis_dir / "classic_seed_summary.tex"
subprocess.run(
    [
        sys.executable,
        "-m",
        "baselines.analysis.make_classic_latex_table",
        "--input",
        str(EXPERIMENT_DIR / "classic_ml_results.csv"),
        "--output",
        str(classic_fastest_table),
        "--full-output",
        str(classic_full_table),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "baselines.analysis.make_seed_summary_latex_table",
        "--input",
        str(EXPERIMENT_DIR / "per_seed_report_analysis.csv"),
        "--output",
        str(classic_seed_summary_table),
        "--group-by",
        "family",
        "--caption",
        "Classic baseline validation metrics across five seeds.",
        "--label",
        "tab:classic-seed-summary",
    ],
    check=True,
)

classic_fastest_table, classic_full_table, classic_seed_summary_table


Wrote experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/analysis/classic_fastest_families.tex
Wrote experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/analysis/classic_full_results.tex


Wrote experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/analysis/classic_seed_summary.tex


(PosixPath('experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/analysis/classic_fastest_families.tex'),
 PosixPath('experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/analysis/classic_full_results.tex'),
 PosixPath('experiments/classic_ml/20260524_100658_CLASSIC_ML_BASELINES/analysis/classic_seed_summary.tex'))